### Analysis of trained MICrONS model 

In [ ]:
from caveclient import CAVEclient
import pandas as pd

import datajoint as dj
# dj.config['database.host'] = 'localhost'
# dj.config['database.port'] = 3307
# dj.config['database.user'] = os.environ['DJ_USER']  # REDACTED
# dj.config['database.password'] = os.environ['DJ_PASS']  # REDACTED
from microns_phase3 import nda, utils
import matplotlib.pyplot as plt
import numpy as np

import os, sys
#sys.path.append("/notebooks/workspace/microns/")
#sys.path.append('/home/naomi/Desktop/code/microns_rahul/microns')
#from src import constants
import seaborn as sns
# from src.responses import ResponseNormalizer
#from src.data import get_mouse_data
#from src.metrics import corr

from tqdm import tqdm

In [ ]:
# Set plot context and style for consistency
sns.set_context(
    "notebook", font_scale=1.0
)  # Base font scale, can adjust further if needed
plt.rcParams.update(
    {"axes.labelsize": 13, "xtick.labelsize": 10, "ytick.labelsize": 10}
)

In [ ]:
client = CAVEclient()
client = CAVEclient('minnie65_public')
client.materialize.version = 1181

In [ ]:
matched_df = client.materialize.query_table('coregistration_manual_v4')
matched_df.head()

In [ ]:
import pickle as pkl
pkl.dump(matched_df,open('/data1/rb1/microns/saves/matched_df.pkl','wb'))

In [ ]:
area = client.materialize.query_table('nucleus_functional_area_assignment')
area.head()

In [ ]:
pkl.dump(area,open('/data1/rb1/microns/saves/area.pkl','wb'))

In [ ]:
synapse1 = {}
from tqdm import tqdm
for preid in tqdm(np.unique(matched_df.pt_root_id.values)):
    synapse1[preid] = client.materialize.synapse_query(pre_ids=preid)

In [ ]:
counts

In [ ]:
func_df = client.materialize.query_table('functional_properties_v3_bcm')

In [ ]:
experiment = "true_batch_001_clip"
prediction_dir = constants.predictions_dir / experiment / "out-of-fold"
correlations = dict()
dataset = "new"

constants.sensorium_dir

In [ ]:
synapse1 = pkl.load(open('/data1/rb1/microns/saves/synapses_matcheddf_frompre.pkl','rb'))
synapse2 = {}
for preid in tqdm(np.unique(matched_df.pt_root_id.values)):
    for postid in np.unique(matched_df.pt_root_id.values):
        synapse2[(preid,postid)] = np.sum(synapse1[preid].post_pt_root_id.values == postid)

In [ ]:
# 1) Flatten the dict into one DataFrame of observed edges
import pickle as pkl
synapse1 = pkl.load(open('/data1/rb1/microns/saves/synapses_matcheddf_frompre.pkl','rb'))
recs = []
for pre, df in synapse1.items():
    if "post_pt_root_id" not in df.columns or len(df) == 0:
        continue
    recs.append(pd.DataFrame({"pre": pre, "post": df["post_pt_root_id"].values}))
syn_df = pd.concat(recs, ignore_index=True)

# 2) (Optional) restrict to the node set you care about
nodes = np.unique(matched_df.pt_root_id.values)
nodes_set = set(nodes)
syn_df = syn_df[syn_df["pre"].isin(nodes_set) & syn_df["post"].isin(nodes_set)]

# 3) Count synapses per (pre, post)
# pandas >=1.5: value_counts on columns
counts = syn_df.value_counts(["pre","post"]).sort_index()

# Fast lookup helper
def syn_count(preid, postid):
    try:
        return int(counts.loc[(preid, postid)])
    except KeyError:
        return 0

In [ ]:
counts.to_excel("/data1/rb1/microns/saves/counts.xlsx", index=False)

In [ ]:
pkl.dump(counts,open('/data1/rb1/microns/saves/counts.pkl','wb'))

In [ ]:
import pandas as pd

# Load Excel file
counts = pd.read_excel("/data1/rb1/microns/saves/counts.xlsx")

# View first few rows
print(counts.head())


In [ ]:
synapsesfound = list(counts.keys())

In [ ]:
proof_df = client.materialize.tables.proofreading_status_and_strategy(status_axon='t').query(desired_resolution=[1, 1, 1], split_positions=True)
proof_df["strategy_axon"].value_counts()

In [ ]:
proved_nodes = np.unique(proof_df.pt_root_id.values)

In [ ]:
# assuming:
# synapsesfound = [(pre1, post1), (pre2, post2), ...]
# proof_nodes = [id1, id2, id3, ...]

proof_nodes = set(proved_nodes)  # convert to set for O(1) lookups

filtered_synapses = [
    (pre, post)
    for pre, post in synapsesfound
    if pre in proof_nodes and post in proof_nodes
]

print(len(filtered_synapses), "pairs kept out of", len(synapsesfound))


In [ ]:
pkl.dump(filtered_synapses, open('/data1/rb1/microns/saves/filtered_synapses.pkl','wb'))

In [ ]:
pkl.dump(func_df,open('/data1/rb1/microns/saves/func_df.pkl','wb'))

In [ ]:
#from caveclient import CAVEclient
import pandas as pd

#client = CAVEclient("<your_datastack>")   # e.g., "microns_mm3" (use whatever you already use)

def pt_to_area(pt_id):
    df = area
    return df.tag[df.pt_root_id.values == pt_id].values[0]

# example
id1 = 864691135661391856
areas = pt_to_area(id1)
print(areas)


In [ ]:
func_df = client.materialize.query_table('functional_properties_v3_bcm')

In [ ]:
import pickle as pkl

In [ ]:
filtered_synapses = pkl.load(open('/data1/rb1/microns/saves/filtered_synapses.pkl','rb'))
func_df = pkl.load(open('/data1/rb1/microns/saves/func_df.pkl','rb'))
matched_df = pkl.load(open('/data1/rb1/microns/saves/matched_df.pkl','rb'))
area = pkl.load(open('/data1/rb1/microns/saves/area.pkl','rb'))

In [ ]:
func_df.gOSI.values[func_df.pt_root_id.values == filtered_synapses[0][0]]
func_df.columns

In [ ]:
np.mean(func_df.gOSI.values[func_df.pt_root_id.values == filtered_synapses[0][1]])-np.mean(func_df.gOSI.values[func_df.pt_root_id.values == filtered_synapses[0][0]])

def find_meanDelta(func_df, synapses):
    gosi_synapses = np.nanmean([np.abs(np.mean(func_df.gOSI.values[func_df.pt_root_id.values == j][func_df.gOSI.values[func_df.pt_root_id.values == j]>0.25])-np.mean(func_df.gOSI.values[func_df.pt_root_id.values == i][func_df.gOSI.values[func_df.pt_root_id.values == i]>0.25])) if (i in func_df.pt_root_id.values) and (j in func_df.pt_root_id.values) and (pt_to_area(i) == 'V1') and (pt_to_area(j) == 'V1') else np.nan for (i,j) in synapses])
    gdsi_synapses = np.nanmean([np.abs(np.mean(func_df.gDSI.values[func_df.pt_root_id.values == j][func_df.gOSI.values[func_df.pt_root_id.values == j]>0.25])-np.mean(func_df.gDSI.values[func_df.pt_root_id.values == i][func_df.gOSI.values[func_df.pt_root_id.values == i]>0.25])) if (i in func_df.pt_root_id.values) and (j in func_df.pt_root_id.values) and (pt_to_area(i) == 'V1') and (pt_to_area(j) == 'V1') else np.nan for (i,j) in synapses])
    gprefori_synapses = np.nanmean([np.abs(np.mean(func_df.pref_ori.values[func_df.pt_root_id.values == j][func_df.gOSI.values[func_df.pt_root_id.values == j]>0.25])-np.mean(func_df.pref_ori.values[func_df.pt_root_id.values == i][func_df.gOSI.values[func_df.pt_root_id.values == i]>0.25])) if (i in func_df.pt_root_id.values) and (j in func_df.pt_root_id.values) and (pt_to_area(i) == 'V1') and (pt_to_area(j) == 'V1') else np.nan for (i,j) in synapses])
    gprefdir_synapses = np.nanmean([np.abs(np.mean(func_df.pref_dir.values[func_df.pt_root_id.values == j][func_df.gOSI.values[func_df.pt_root_id.values == j]>0.25])-np.mean(func_df.pref_dir.values[func_df.pt_root_id.values == i][func_df.gOSI.values[func_df.pt_root_id.values == i]>0.25])) if (i in func_df.pt_root_id.values) and (j in func_df.pt_root_id.values) and (pt_to_area(i) == 'V1') and (pt_to_area(j) == 'V1') else np.nan for (i,j) in synapses])
    return {'d_gosi':gosi_synapses, 'd_gdsi':gdsi_synapses, 'd_gprefori':gprefori_synapses,'d_gprefdir':gprefdir_synapses}

def get_distUnconn(unconnpairs, niter, n_synapses, seed = 0):
    unconnpairs_mean_list = []
    rng = np.random.default_rng(seed)
    for _ in tqdm(range(niter)):
        perm = rng.permutation(len(unconnpairs))
        idxlist = perm[:n_synapses]
        unconnpairs_rand = [unconnpairs[idx] for idx in idxlist]
        unconnpairs_mean = find_meanDelta(func_df, unconnpairs_rand)
        unconnpairs_mean_list.append(unconnpairs_mean)
    return unconnpairs_mean_list

meanconn = find_meanDelta(func_df, filtered_synapses)
print(meanconn)
n_synapses = len(filtered_synapses)
print(n_synapses)

print('start')
#from itertools import combinations
all_nodes = proved_nodes

nodes = np.asarray(all_nodes, dtype=np.int64)
I, J = np.meshgrid(nodes, nodes, indexing='ij')
mask = (I != J)
pairs = np.stack((I[mask], J[mask]), axis=1)  # shape (N*(N-1), 2)

filt = np.asarray(list({(int(a), int(b)) for a, b in filtered_synapses if a != b}),
                  dtype=np.int64)

# view as structured to use setdiff1d on rows
pairs_view = pairs.view([('a', np.int64), ('b', np.int64)]).ravel()
filt_view  = filt.view([('a', np.int64), ('b', np.int64)]).ravel()

keep = np.setdiff1d(pairs_view, filt_view, assume_unique=False)
others = keep.view(np.int64).reshape(-1, 2)
othersV1 = []
for idx in tqdm(range(len(others))):
    if others[idx][0] in area.pt_root_id.values and others[idx][1] in area.pt_root_id.values:
        if (pt_to_area(others[idx][0])=='V1') and (pt_to_area(others[idx][1])=='V1'):
            othersV1.append(others[idx])
print('done')

niter = 2000
distmeanunconn = get_distUnconn(othersV1, niter, n_synapses)
pkl.dump(distmeanunconn,open('/data1/rb1/microns/saves/hosi_distmeanunconn.pkl','wb'))

In [ ]:
pt_to_area(others[idx][1])

In [ ]:
area.tag[area.pt_root_id.values == 864691135279153313].values[0]

In [ ]:
np.random.default_rng(42).permutation(10)

In [ ]:
print(distmeanunconn[0].keys())
distmeanunconn_dgosi = [distmeanunconn[i]['d_gosi'] for i in range(len(distmeanunconn))]
distmeanunconn_dgdsi = [distmeanunconn[i]['d_gdsi'] for i in range(len(distmeanunconn))]
distmeanunconn_dprefori = [distmeanunconn[i]['d_gprefori'] for i in range(len(distmeanunconn))]
distmeanunconn_dgprefdir = [distmeanunconn[i]['d_gprefdir'] for i in range(len(distmeanunconn))]

obs = {"gOSI": meanconn['d_gosi'], "gDSI": meanconn['d_gdsi'], "PrefOri": meanconn['d_gprefori'], "PrefDir": meanconn['d_gprefdir']}

# plt.hist(distmeanunconn_dgosi)
# plt.show()
# plt.hist(distmeanunconn_dgdsi)
# plt.show()
# plt.hist(distmeanunconn_dprefori)
# plt.show()
# plt.hist(distmeanunconn_dgprefdir)
# plt.show()

distall = {"gOSI": distmeanunconn_dgosi, "gDSI": distmeanunconn_dgdsi, "PrefOri": distmeanunconn_dprefori, "PrefDir": distmeanunconn_dgprefdir}

#print(distall[0])
import matplotlib.pyplot as plt, os
os.makedirs("figs_nullplots", exist_ok=True)

pct = {m: 100.0 * np.nanmean(distall[m] <= obs[m]) for m in obs.keys()}

for idx,m in enumerate(["gOSI","gDSI","PrefOri","PrefDir"]):
    nm = np.array(distall[m])
    plt.figure(figsize=(8,4))
    plt.hist(nm[~np.isnan(nm)], bins=40, alpha=0.7, label="Null mean (matched)")
    plt.axvline(obs[m], linestyle="--", linewidth=2, label=f"Connected mean = {obs[m]:.3f}")
    p = pct[m]
    label = "|ΔPrefDir|" if m=="PrefDir" else ("|ΔPrefOri|" if m=="PrefOri" else f"|Δ{m}|")
    plt.title(f"{label}: connected mean at {p:.1f}th percentile of null")
    plt.xlabel(f"Mean {label}"); plt.ylabel("Count"); plt.legend(); plt.tight_layout()
    plt.savefig(f"figs_nullplots/null_mean_{m}.png", dpi=300); plt.close()

In [ ]:
nodes = np.asarray(all_nodes, dtype=np.int64)
I, J = np.meshgrid(nodes, nodes, indexing='ij')
mask = (I != J)
pairs = np.stack((I[mask], J[mask]), axis=1)

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# ---------- helpers ----------
def _as_tuple_pos(v):
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) == 3:
        return (float(v[0]), float(v[1]), float(v[2]))
    return None

def _circ_delta_deg(a, b, period):
    # shortest signed circular difference on [0, period)
    return (a - b + period/2) % period - period/2

def _quantile_bins(vals, q=10):
    vals = np.asarray(vals, dtype=float)
    good = np.isfinite(vals)
    if good.sum() == 0 or len(np.unique(vals[good])) <= 1:
        return np.zeros_like(vals, dtype=int), np.array([-np.inf, np.inf])
    edges = np.quantile(vals[good], np.linspace(0, 1, q+1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)
    if len(edges) < 3:
        edges = np.array([-np.inf, np.inf])
    bins = np.digitize(vals, edges[1:-1], right=False)
    return bins, edges

# ---------- prep (once) ----------
def prepare_for_null(
    synapsesfound, func_df, directed=False, distance_bins=10, seed=0, match_by_distance=True
):
    rng = np.random.default_rng(seed)

    # required columns
    for c in ["pt_root_id", "gOSI", "gDSI"]:
        if c not in func_df.columns:
            raise ValueError(f"func_df missing {c}")
    fdf = func_df.copy()
    ids = fdf["pt_root_id"].astype(int).values
    U = set(int(x) for x in ids)

    # maps
    gosi = dict(zip(ids, fdf["gOSI"].astype(float)))
    gdsi = dict(zip(ids, fdf["gDSI"].astype(float)))
    pref_dir = dict(zip(ids, fdf["pref_dir"].astype(float))) if "pref_dir" in fdf.columns else {}
    pref_ori = dict(zip(ids, fdf["pref_ori"].astype(float))) if "pref_ori" in fdf.columns else {}

    # positions
    pos = {}
    if "pt_position" in fdf.columns:
        for _, row in fdf[["pt_root_id","pt_position"]].dropna(subset=["pt_position"]).iterrows():
            p = _as_tuple_pos(row["pt_position"])
            if p is not None:
                pos[int(row["pt_root_id"])] = p

    # connected (unique)
    if directed:
        C = {(int(a),int(b)) for a,b in synapsesfound if int(a) in U and int(b) in U and int(a)!=int(b)}
    else:
        C = {tuple(sorted((int(a),int(b)))) for a,b in synapsesfound if int(a) in U and int(b) in U and int(a)!=int(b)}
    C_list = list(C)

    def dist(a,b):
        pa, pb = pos.get(a), pos.get(b)
        if (pa is None) or (pb is None): return np.nan
        return float(np.linalg.norm(np.array(pa)-np.array(pb)))

    # distance bins from connected
    use_dist = bool(match_by_distance and len(pos) >= 2)
    if use_dist:
        C_dists = np.array([dist(a,b) for (a,b) in C_list], dtype=float)
        good = np.isfinite(C_dists)
        if good.sum() < max(10, 0.5*len(C_list)):
            use_dist = False
        else:
            C_bins, edges = _quantile_bins(C_dists, q=distance_bins)
    if not use_dist:
        C_bins = np.zeros(len(C_list), dtype=int)
        edges = np.array([-np.inf, np.inf])

    need_per_bin = defaultdict(int)
    for b in C_bins:
        need_per_bin[int(b)] += 1

    return dict(
        units=np.array(sorted(U), dtype=int),
        C_list=C_list,
        C_set=set(C),
        gosi=gosi, gdsi=gdsi,
        pref_dir=pref_dir, pref_ori=pref_ori,
        pos=pos,
        use_dist=use_dist, edges=edges, need_per_bin=need_per_bin,
        rng=rng, directed=directed
    )

# ---------- one replicate from full null ----------
def draw_null_replicate(ctx, metric="gOSI", max_tries_per_pair=2000):
    units = ctx["units"]; C_set = ctx["C_set"]; rng = ctx["rng"]; directed = ctx["directed"]
    gosi, gdsi = ctx["gosi"], ctx["gdsi"]
    pref_dir, pref_ori = ctx["pref_dir"], ctx["pref_ori"]
    use_dist, edges, need_per_bin = ctx["use_dist"], ctx["edges"], ctx["need_per_bin"]
    pos = ctx["pos"]

    def dist(a,b):
        pa, pb = pos.get(a), pos.get(b)
        if (pa is None) or (pb is None): return np.nan
        return float(np.linalg.norm(np.array(pa)-np.array(pb)))

    picks, used = [], set()
    for b, k in need_per_bin.items():
        got, tries = [], 0
        lo, hi = edges[b], edges[b+1]
        while len(got) < k and tries < max_tries_per_pair*k:
            a, c = rng.choice(units, size=2, replace=False)
            p = (a,c) if directed else tuple(sorted((a,c)))
            if p in C_set or p in used or p in got or a==c:
                tries += 1; continue
            if use_dist:
                d = dist(a,c)
                if not np.isfinite(d) or not (lo < d <= hi):
                    tries += 1; continue
            got.append(p); tries += 1
        while len(got) < k:
            a, c = rng.choice(units, size=2, replace=False)
            p = (a,c) if directed else tuple(sorted((a,c)))
            if p in C_set or p in used or p in got or a==c:
                continue
            got.append(p)
        picks.extend(got); used.update(got)

    vals = []
    for a,b in picks:
        if metric == "gOSI":
            va, vb = gosi.get(a,np.nan), gosi.get(b,np.nan)
            if np.isnan(va) or np.isnan(vb): continue
            vals.append(abs(va - vb))
        elif metric == "gDSI":
            va, vb = gdsi.get(a,np.nan), gdsi.get(b,np.nan)
            if np.isnan(va) or np.isnan(vb): continue
            vals.append(abs(va - vb))
        elif metric == "PrefDir":
            pa, pb = pref_dir.get(a,np.nan), pref_dir.get(b,np.nan)
            if np.isnan(pa) or np.isnan(pb): continue
            vals.append(abs(_circ_delta_deg(pa, pb, 360.0)))
        elif metric == "PrefOri":
            oa, ob = pref_ori.get(a,np.nan), pref_ori.get(b,np.nan)
            if np.isnan(oa) or np.isnan(ob): continue
            vals.append(abs(_circ_delta_deg(oa, ob, 180.0)))
        else:
            raise ValueError("metric must be in {'gOSI','gDSI','PrefDir','PrefOri'}")
    return float(np.mean(vals)) if len(vals) else np.nan

# ---------- null distribution (many replicates) ----------
def null_distribution_of_mean(ctx, metric="gOSI", n_random=2000, seed=None):
    if seed is not None:
        ctx = dict(ctx); ctx["rng"] = np.random.default_rng(seed)
    iterator = range(n_random)
    if tqdm is not None:
        iterator = tqdm(iterator, desc=f"Null reps: {metric}", unit="rep")
    means = [draw_null_replicate(ctx, metric=metric) for _ in iterator]
    return np.array(means)

# ---------- observed mean on connected ----------
def observed_connected_mean(ctx, metric="gOSI"):
    gosi, gdsi = ctx["gosi"], ctx["gdsi"]
    pref_dir, pref_ori = ctx["pref_dir"], ctx["pref_ori"]
    vals = []
    for a,b in ctx["C_list"]:
        if metric == "gOSI":
            va, vb = gosi.get(a,np.nan), gosi.get(b,np.nan)
            if np.isnan(va) or np.isnan(vb): continue
            vals.append(abs(va - vb))
        elif metric == "gDSI":
            va, vb = gdsi.get(a,np.nan), gdsi.get(b,np.nan)
            if np.isnan(va) or np.isnan(vb): continue
            vals.append(abs(va - vb))
        elif metric == "PrefDir":
            pa, pb = pref_dir.get(a,np.nan), pref_dir.get(b,np.nan)
            if np.isnan(pa) or np.isnan(pb): continue
            vals.append(abs(_circ_delta_deg(pa, pb, 360.0)))
        elif metric == "PrefOri":
            oa, ob = pref_ori.get(a,np.nan), pref_ori.get(b,np.nan)
            if np.isnan(oa) or np.isnan(ob): continue
            vals.append(abs(_circ_delta_deg(oa, ob, 180.0)))
        else:
            raise ValueError("metric must be in {'gOSI','gDSI','PrefDir','PrefOri'}")
    return float(np.mean(vals)) if len(vals) else np.nan


In [ ]:
# 1) Prep once (no single df_null; we resample fresh each replicate)
ctx = prepare_for_null(
    synapsesfound=filtered_synapses,
    func_df=func_df,              # has pt_root_id, gOSI, gDSI, pref_dir, pref_ori, pt_position
    directed=True,
    distance_bins=10,
    seed=0,
    match_by_distance=False,       # or False for uniform null
)

# 2) Observed means (connected pairs)
obs = {m: observed_connected_mean(ctx, metric=m) for m in ["gOSI","gDSI","PrefDir","PrefOri"]}

# 3) Null distributions (fresh matched-null each replicate)
null = {m: null_distribution_of_mean(ctx, metric=m, n_random=5000, seed=1) for m in obs.keys()}

# 4) Percentiles (one-sided lower tail; flip sign if testing the other direction)
pct = {m: 100.0 * np.nanmean(null[m] <= obs[m]) for m in obs.keys()}
print(pct)

# 5) Plot example
import matplotlib.pyplot as plt, os
os.makedirs("figs_nullplots", exist_ok=True)
for m in ["gOSI","gDSI","PrefDir","PrefOri"]:
    nm = null[m]
    plt.figure(figsize=(8,4))
    plt.hist(nm[~np.isnan(nm)], bins=40, alpha=0.7, label="Null mean (matched)")
    plt.axvline(obs[m], linestyle="--", linewidth=2, label=f"Connected mean = {obs[m]:.3f}")
    p = pct[m]
    label = "|ΔPrefDir|" if m=="PrefDir" else ("|ΔPrefOri|" if m=="PrefOri" else f"|Δ{m}|")
    plt.title(f"{label}: connected mean at {p:.1f}th percentile of null")
    plt.xlabel(f"Mean {label}"); plt.ylabel("Count"); plt.legend(); plt.tight_layout()
    plt.savefig(f"figs_nullplots/null_mean_{m}.png", dpi=300); plt.close()


In [ ]:
# 5) Plot example
import matplotlib.pyplot as plt, os
os.makedirs("figs_nullplots", exist_ok=True)
for m in ["gOSI","gDSI","PrefDir","PrefOri"]:
    nm = null[m]
    plt.figure(figsize=(8,4))
    plt.hist(nm[~np.isnan(nm)], bins=40, alpha=0.7, label="Null mean")
    plt.axvline(obs[m], linestyle="--", linewidth=2, label=f"Connected mean = {obs[m]:.3f}")
    p = pct[m]
    label = "|ΔPrefDir|" if m=="PrefDir" else ("|ΔPrefOri|" if m=="PrefOri" else f"|Δ{m}|")
    plt.title(f"{label}: connected mean at {p:.1f}th percentile of null")
    plt.xlabel(f"Mean {label}"); plt.ylabel("Count"); plt.legend(); plt.tight_layout()
    plt.savefig(f"figs_nullplots/null_mean_{m}.png", dpi=300); plt.close()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from itertools import combinations
from collections import defaultdict

# --------------------------------------------
# Utilities
# --------------------------------------------
def _as_tuple_pos(v):
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) == 3:
        return (float(v[0]), float(v[1]), float(v[2]))
    return None

def _circ_delta_deg(a, b, period=360.0):
    # shortest signed circular difference
    return (a - b + period/2) % period - period/2

def _quantile_bins(vals, q=10):
    vals = np.asarray(vals, dtype=float)
    good = np.isfinite(vals)
    if good.sum() == 0 or len(np.unique(vals[good])) <= 1:
        edges = np.array([-np.inf, np.inf])
        bins = np.zeros_like(vals, dtype=int)
        return bins, edges
    edges = np.quantile(vals[good], np.linspace(0, 1, q+1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)
    if len(edges) < 3:
        edges = np.array([-np.inf, np.inf])
    bins = np.digitize(vals, edges[1:-1], right=False)
    return bins, edges

# --------------------------------------------
# Core: build connected vs matched-null pairs (with progress)
# --------------------------------------------
def build_connected_and_matched_null_simple(
    synapsesfound,
    func_df,                 # must include: pt_root_id, gOSI, gDSI; optional: pref_dir, pt_position
    distance_bins=10,
    directed=False,
    seed=0,
    verbose=True,
):
    rng = np.random.default_rng(seed)

    # Maps from pt_root_id
    if "pt_root_id" not in func_df or "gOSI" not in func_df or "gDSI" not in func_df:
        raise ValueError("func_df must contain pt_root_id, gOSI, gDSI.")
    fdf = func_df.copy()
    ids = fdf["pt_root_id"].astype(int).values
    gosi_map = dict(zip(ids, fdf["gOSI"].astype(float)))
    gdsi_map = dict(zip(ids, fdf["gDSI"].astype(float)))
    pref_map = dict(zip(
        ids,
        fdf["pref_dir"].astype(float) if "pref_dir" in fdf.columns else pd.Series(index=fdf.index, dtype=float)
    ))

    # Position map (optional)
    pos_map = {}
    if "pt_position" in fdf.columns:
        for _, row in fdf[["pt_root_id", "pt_position"]].dropna(subset=["pt_position"]).iterrows():
            p = _as_tuple_pos(row["pt_position"])
            if p is not None:
                pos_map[int(row["pt_root_id"])] = p

    U = set(int(x) for x in ids)

    # Connected pairs (filtered to U)
    if directed:
        C = {(int(a), int(b)) for a, b in synapsesfound if int(a) in U and int(b) in U and int(a) != int(b)}
    else:
        C = {tuple(sorted((int(a), int(b)))) for a, b in synapsesfound if int(a) in U and int(b) in U and int(a) != int(b)}
    C_list = list(C)

    # Distance helper
    def dist(a, b):
        pa, pb = pos_map.get(a), pos_map.get(b)
        if (pa is None) or (pb is None):
            return np.nan
        return float(np.linalg.norm(np.array(pa) - np.array(pb)))

    # Distances for connected
    C_dists = np.array([dist(a, b) for (a, b) in C_list], dtype=float)
    have_dists = np.isfinite(C_dists).sum() >= max(10, 0.5 * len(C_dists))
    if have_dists:
        C_bins, edges = _quantile_bins(C_dists, q=distance_bins)
    else:
        C_bins = np.zeros(len(C_list), dtype=int)
        edges = np.array([-np.inf, np.inf])

    # Group connected pairs by distance-bin
    C_by_bin = defaultdict(list)
    for idx, lab in enumerate(C_bins):
        C_by_bin[int(lab)].append(idx)

    # Usable units
    if have_dists:
        usable_units = sorted({u for u in U if u in pos_map})
    else:
        usable_units = sorted(U)

    # All candidate pairs minus connected
    if not directed:
        all_pairs = {tuple(sorted(p)) for p in combinations(usable_units, 2)}
    else:
        all_pairs = {(i, j) for i in usable_units for j in usable_units if i != j}
    candidates = list(all_pairs - C)

    # Bin each candidate by distance
    def bin_of_pair(p):
        a, b = p
        if not have_dists:
            return 0
        d = dist(a, b)
        if not np.isfinite(d):
            return None
        return int(np.digitize([d], edges[1:-1], right=False)[0])

    pool_by_bin = defaultdict(list)
    for p in candidates:
        b = bin_of_pair(p)
        if b is not None:
            pool_by_bin[b].append(p)

    # ------- Progress: header -------
    if verbose:
        print("=== Matched-null sampling ===")
        print(f"Units with metrics: {len(U):,}")
        print(f"Connected pairs (unique, {'directed' if directed else 'undirected'}): {len(C_list):,}")
        print(f"Positions available: {len(pos_map):,} units  -> distance strata: {'ON' if have_dists else 'OFF'}")
        if have_dists:
            print(f"Distance bins requested: {distance_bins}  |  actual bins present: {len({int(b) for b in C_bins})}")
        print(f"Candidate non-edges: {len(candidates):,}")
        # quick per-bin summary
        if have_dists:
            print("Bin  | #Connected | #Candidates")
            for b in sorted(C_by_bin.keys()):
                print(f"{b:>3}  | {len(C_by_bin[b]):>10} | {len(pool_by_bin.get(b, [])):>11}")
        print("-" * 46)

    # Sample matched nulls per bin
    N_pairs = []
    used = set()
    total_need = len(C_list)
    done = 0
    for b, idxs in sorted(C_by_bin.items()):
        need = len(idxs)
        pool = pool_by_bin.get(b, [])
        if len(pool) >= need:
            picks = rng.choice(pool, size=need, replace=False).tolist()
        else:
            picks = list(pool)
            short = need - len(picks)
            remaining = list(set(candidates) - set(used) - set(picks))
            short = min(short, len(remaining))
            if short > 0:
                picks += rng.choice(remaining, size=short, replace=False).tolist()
        N_pairs.extend(picks)
        used.update(picks)
        done += need
        if verbose:
            print(f"[bin {b}] sampled {len(picks)}/{need}  |  progress {done}/{total_need} pairs")

    if verbose:
        print("-" * 46)
        print(f"Matched-null total: {len(N_pairs):,}  (target {len(C_list):,})")

    # Build outputs
    def pairs_to_df(pairs, label):
        rows = []
        for a, b in pairs:
            ga, gb = gosi_map.get(a, np.nan), gosi_map.get(b, np.nan)
            da, db = gdsi_map.get(a, np.nan), gdsi_map.get(b, np.nan)
            if np.isnan(ga) or np.isnan(gb) or np.isnan(da) or np.isnan(db):
                continue
            pa, pb = pref_map.get(a, np.nan), pref_map.get(b, np.nan)
            dtheta = np.nan if (np.isnan(pa) or np.isnan(pb)) else abs(_circ_delta_deg(pa, pb, 360.0))
            rows.append({
                "pre": a, "post": b,
                "abs_d_gOSI": abs(ga - gb),
                "abs_d_gDSI": abs(da - db),
                "abs_d_PrefDir": dtheta,
                "mean_gOSI": 0.5 * (ga + gb),
                "mean_gDSI": 0.5 * (da + db),
                "set": label
            })
        return pd.DataFrame(rows)

    df_conn = pairs_to_df(C_list, "connected")
    df_null = pairs_to_df(N_pairs, "null_matched")

    if verbose:
        print(f"Output rows -> connected: {len(df_conn):,}, null: {len(df_null):,}")

    return df_conn, df_null

# --------------------------------------------
# Plotting: null-of-mean vs observed (simple)
# --------------------------------------------
def plot_null_vs_observed(
    df_conn,
    df_null,
    metrics=("abs_d_gOSI", "abs_d_gDSI", "abs_d_PrefDir"),
    n_random=1000,
    bins=40,
    seed=0,
    outdir="figs_nullplots",
):
    os.makedirs(outdir, exist_ok=True)
    rng = np.random.default_rng(seed)

    for metric in metrics:
        if metric not in df_conn.columns or metric not in df_null.columns:
            continue
        x_conn = df_conn[metric].dropna().values
        x_null = df_null[metric].dropna().values
        if len(x_conn) == 0 or len(x_null) < len(x_conn):
            continue
        n = len(x_conn)
        obs_mean = float(np.mean(x_conn))
        null_means = np.array([np.mean(rng.choice(x_null, size=n, replace=False))
                               for _ in range(n_random)])
        percentile = 100.0 * (null_means <= obs_mean).mean()

        plt.figure(figsize=(8, 4))
        plt.hist(null_means, bins=bins, alpha=0.7, label="Null mean (unconnected)")
        plt.axvline(obs_mean, linestyle="--", linewidth=2,
                    label=f"Connected mean = {obs_mean:.3f}")
        plt.title(f"{metric}: connected mean at {percentile:.1f}th percentile of null")
        plt.xlabel(f"Mean {metric}")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f"null_{metric}.png"), dpi=300)
        plt.close()
    print(f"✅ Plots saved to {outdir}")

# --------------------------------------------
# Example usage
# --------------------------------------------
df_conn, df_null = build_connected_and_matched_null_simple(
    synapsesfound=filtered_synapses,
    func_df=func_df,          # pt_root_id, gOSI, gDSI; optional: pref_dir, pt_position
    distance_bins=10,
    directed=True,           # set True only if you truly want directed pairs
    seed=0,
    verbose=True,             # <<< progress prints
)
df_conn.to_pickle("connected_pairs_metrics.pkl")
df_null.to_pickle("null_pairs_metrics.pkl")
plot_null_vs_observed(df_conn, df_null,
                      metrics=("abs_d_gOSI","abs_d_gDSI","abs_d_PrefDir"),
                      n_random=2000, bins=40, seed=0)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# Optional: add |Δ PrefDir| to pair tables if func_df has it
# ------------------------------------------------------------
def _circ_delta_deg(a, b, period=360.0):
    # shortest signed difference on a circle
    d = (a - b + period/2) % period - period/2
    return d

def add_prefdir_delta(df_pairs, func_df, prefdir_col="Preferred_Direction"):
    if prefdir_col not in func_df.columns:
        return df_pairs  # nothing to do
    ang = dict(zip(func_df["pt_root_id"].astype(int), func_df[prefdir_col].astype(float)))
    rows = []
    for _, r in df_pairs.iterrows():
        a, b = int(r["pre"]), int(r["post"])
        pa, pb = ang.get(a, np.nan), ang.get(b, np.nan)
        if np.isnan(pa) or np.isnan(pb):
            rows.append(np.nan)
        else:
            rows.append(abs(_circ_delta_deg(pa, pb, 360.0)))
    df_pairs = df_pairs.copy()
    df_pairs["abs_d_PrefDir"] = rows
    return df_pairs

# ------------------------------------------------------------
# Core plotting: null (unconnected) resamples vs. observed mean
# ------------------------------------------------------------
def plot_null_vs_observed(
    df_conn,
    df_null,
    metrics=("abs_d_gOSI","abs_d_gDSI","abs_d_PrefDir"),
    n_random=1000,
    bins=30,
    seed=42,
    outdir="figs_nullplots"
):
    os.makedirs(outdir, exist_ok=True)
    rng = np.random.default_rng(seed)

    for metric in metrics:
        if metric not in df_conn.columns:
            print(f"Skip {metric}: not in df_conn.")
            continue
        if metric not in df_null.columns:
            print(f"Skip {metric}: not in df_null.")
            continue

        x_conn = df_conn[metric].dropna().values
        x_null = df_null[metric].dropna().values
        if len(x_conn) == 0 or len(x_null) < len(x_conn):
            print(f"Skip {metric}: insufficient data (conn={len(x_conn)}, null={len(x_null)}).")
            continue

        n = len(x_conn)
        obs_mean = float(np.mean(x_conn))
        # bootstrap null of mean with *no replacement* to mirror matched size
        null_means = np.array([np.mean(rng.choice(x_null, size=n, replace=False)) for _ in range(n_random)])

        # percentile of observed vs null
        percentile = 100.0 * (null_means <= obs_mean).mean()

        # (optional) simple effect size
        pooled_sd = np.sqrt(((x_conn.var(ddof=1) + x_null.var(ddof=1)) / 2.0))
        cohens_d = (x_conn.mean() - x_null.mean()) / (pooled_sd + 1e-12)

        plt.figure(figsize=(8, 4))
        plt.hist(null_means, bins=bins, alpha=0.7, label="Null mean (unconnected)")
        plt.axvline(obs_mean, linestyle="--", linewidth=2,
                    label=f"Observed mean (connected) = {obs_mean:.3f}")
        plt.title(f"{metric}: observed at {percentile:.1f}th percentile of null\nCohen's d vs. null = {cohens_d:.2f}")
        plt.xlabel(f"Mean {metric}")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f"null_{metric}.png"), dpi=300)
        plt.close()

    print(f"✅ Plots saved to: {outdir}")

# ============================
# Example usage
# ============================
# Assumes you’ve already created:
#   df_conn, df_null = build_connected_and_matched_null(...)

# If you have Preferred_Direction in func_df, add |Δ PrefDir|
# df_conn = add_prefdir_delta(df_conn, func_df, prefdir_col="Preferred_Direction")
# df_null = add_prefdir_delta(df_null, func_df, prefdir_col="Preferred_Direction")

# Make plots (will auto-skip metrics not present)
plot_null_vs_observed(df_conn, df_null,
                     metrics=("abs_d_gOSI","abs_d_gDSI","abs_d_PrefDir"),
                     n_random=2000, bins=40, seed=0, outdir="figs_nullplots")


In [ ]:
import pickle as pkl
pkl.dump(synapse1,open('/data1/rb1/microns/saves/synapses_matcheddf_frompre.pkl','wb'))

In [ ]:
# for exp_id in range(16):
exp_id = 1
mouse = constants.dataset2mice[dataset][exp_id]
mouse_data = get_mouse_data(mouse=mouse, splits=constants.folds_splits)
mouse_prediction_dir = prediction_dir / mouse

def cut_responses_for_submission(prediction: np.ndarray):
    prediction = prediction[..., : constants.submission_limit_length]
    prediction = prediction[..., constants.submission_skip_first :]
    if constants.submission_skip_last:
        prediction = prediction[..., : -constants.submission_skip_last]
    return prediction

predictions = []
targets = []

for trial_data in mouse_data["trials"]:
    trial_data = mouse_data["trials"][0]
    trial_id = trial_data["trial_id"]
    prediction = np.load(str(mouse_prediction_dir / f"{trial_id}.npy"))
    target = np.load(trial_data["response_path"])[..., : trial_data["length"]]
    prediction = cut_responses_for_submission(prediction)
    target = cut_responses_for_submission(target)
    predictions.append(prediction)
    targets.append(target)

correlation = corr(
    np.concatenate(predictions, axis=1), np.concatenate(targets, axis=1), axis=1
)

coords = mouse_data["cell_motor_coordinates"]

df1 = pd.DataFrame({"PredCorr": correlation, "Depth": coords[:, 2]})

g = sns.catplot(x="Depth", y="PredCorr", kind="violin", data=df1)
g.set_axis_labels("Depth (micrometer)", "Predictive Correlation")
savedir = "saves/"
# g.savefig(
#     savedir + "violinplot_predcorr_" + str(exp_id) + ".png",
#     dpi=300,
#     bbox_inches="tight",
# )

### Plotting across depth and cell-type

In [ ]:
#### Creation of matched df that corresponds betweeen anatomical and functional data

#grouping by session and scan_idx
matchedgroupdf = matched_df.groupby(['session','scan_idx'])

#turning into dict
matchedgroupdict = {}
matchedgroupdictsize = {}

for mgroupname, mdf in matchedgroupdf:
    matchedgroupdict[mgroupname] = mdf
    matchedgroupdictsize[mgroupname] = mdf.shape[0]

#just keeping unit_id (unit in func data) and pt_root_id (neuron root in anat data)
unitptrootids = {}
for mgroupname, mdf in matchedgroupdf:
    unitptrootids[mgroupname] = mdf[['unit_id','pt_root_id']]

cell_type_df = client.materialize.query_table('nucleus_detection_v0')
cell_types = client.materialize.query_table('aibs_metamodel_celltypes_v661')

#whichever pt_root_ids in cell_type_df checking and adding celltype_id (another id for celltypes!) into the df
ptrootnucleusid = {}
for key, df in tqdm(unitptrootids.items()):  
    is_subset1 = np.isin(df.pt_root_id.values, cell_type_df.pt_root_id.values)
    is_subset2 = [np.where(cell_type_df.pt_root_id.values == val)[0][0] for val in df.pt_root_id.values[is_subset1]]#np.isin(cell_type_df.pt_root_id.values,df.pt_root_id.values)
    ptrootnucleusid[key] = df.iloc[is_subset1]
    ptrootnucleusid[key]['celltype_id'] = cell_type_df.id.values[is_subset2]


#extracting 'cell_type' ie all cell types
df_ct={}
for key, df in ptrootnucleusid.items():
    acc_celltype = []
    acc_idx = []
    for idx in range(len(df.celltype_id.values)):
        if df.celltype_id.values[idx] in cell_types.id_ref.values:
            acc_celltype.append(cell_types.cell_type.values[np.where(cell_types.id_ref.values==df.celltype_id.values[idx])[0][0]])
            acc_idx.append(idx)
    df_ct[key] = df.iloc[acc_idx].copy()
    df_ct[key]['ctclass'] = acc_celltype

In [ ]:
#### Mapping to get the full cell type name + class

cell_types_mapping = {
    '23P' : ['Excitatory', 'Layer 2/3'],
    '4P' : ['Excitatory', 'Layer 4'],
    '5P-IT' : ['Excitatory', 'Layer 5 intratelencephalic'],
    '5P-ET': ['Excitatory', 'Layer 5 extratelencephalic'],
    '5P-NP' : ['Excitatory', 'Layer 5 near-projecting'],
    '6P-IT' : ['Excitatory', 'Layer 6 intratelencephalic'],
    '6P-CT' : ['Excitatory', 'Layer 6 corticothalamic'],
    'BC' : ['Inhibitory', 'Basket cell'],
    'BPC': ['Inhibitory', 'Bipolar cell'],
    'MC': ['Inhibitory', 'Martinotti cell'],
    'Unsure': ['Inhibitory', 'inhibitory other'],
    'NGC': ['Inhibitory', 'Neurogliaform cell'],
    'microglia': ['Non-neuronal', 'Microglia'],
    'astrocyte': ['Non-neuronal', 'Astrocyte'],
    'OPC': ['Non-neuronal', 'Oligodendroctye precursor cell'],
    'pericyte': ['Non-neuronal', 'Pericyte'],
    'oligo': ['Non-neuronal', 'Oligodendrocyte']
}

In [ ]:
#### Loop through all sessions / scans and plot each individual + create combined df for later plot

mice_ids = df_ct.keys() # get list of possible sessions / scans 
mod_df_ct = pd.DataFrame()

for exp_id in range(16):
    mouse = constants.dataset2mice[dataset][exp_id]
    sess_id, scan_idx = int(mouse.split('-')[1]), int(mouse.split('-')[3])
    
    if (sess_id, scan_idx) in mice_ids:
        print(f'Processing session {sess_id}, scan {scan_idx}')
    else:
        print(f'Missing data for session {sess_id}, scan {scan_idx}')
        continue

    mouse_data = get_mouse_data(mouse=mouse, splits=constants.folds_splits)
    mouse_prediction_dir = prediction_dir / mouse

    def cut_responses_for_submission(prediction: np.ndarray):
        prediction = prediction[..., : constants.submission_limit_length]
        prediction = prediction[..., constants.submission_skip_first :]
        if constants.submission_skip_last:
            prediction = prediction[..., : -constants.submission_skip_last]
        return prediction

    predictions = []
    targets = []

    for trial_data in mouse_data["trials"]:
        trial_data = mouse_data["trials"][0]
        trial_id = trial_data["trial_id"]
        prediction = np.load(str(mouse_prediction_dir / f"{trial_id}.npy"))
        target = np.load(trial_data["response_path"])[..., : trial_data["length"]]
        prediction = cut_responses_for_submission(prediction)
        target = cut_responses_for_submission(target)
        predictions.append(prediction)
        targets.append(target)

    correlation = corr(
        np.concatenate(predictions, axis=1), np.concatenate(targets, axis=1), axis=1)
    
    sub_df_ct = df_ct[(sess_id, scan_idx)]
    sub_df_ct = sub_df_ct.sort_values(by='unit_id')

    unit_ids = sub_df_ct.unit_id.values
    unit_ids -= 1 # adjust to index correlation values correctly 
    corr_values = correlation[unit_ids]

    sub_df_ct['correlation_values'] = corr_values
    sub_df_ct['depth_values'] = mouse_data["cell_motor_coordinates"][unit_ids][:,2]

    cell_class = [cell_types_mapping[x][0] for x in sub_df_ct['ctclass'].values]
    cell_class_spec = [cell_types_mapping[x][1] for x in sub_df_ct['ctclass'].values]

    sub_df_ct['Cell Class'] = cell_class
    sub_df_ct['Cell Type'] = cell_class_spec

    sub_df_ct = sub_df_ct[sub_df_ct['Cell Class'] == 'Excitatory']

    sub_df_ct['session_scan'] = str(sess_id) + "_" + str(scan_idx)
    
    mod_df_ct = pd.concat([mod_df_ct, sub_df_ct])

    # g = sns.catplot(
    #     y="depth_values", 
    #     x="correlation_values", 
    #     hue='Cell Type', 
    #     orient='h', 
    #     kind="strip", 
    #     # dodge=True, 
    #     alpha=0.5,
    #     s=35,
    #     jitter=0.3,
    #     data=sub_df_ct)
    # g.set_axis_labels("Predictive Correlation", "Depth")
    # plt.title(f'Session {sess_id}, Scan {scan_idx}')
    # savedir = "/home/naomi/Desktop/code/microns_rahul/saves/"
    # g.savefig(
    #     savedir + "boxplot_predcorr_celltype_sess" + str(sess_id) + "_scan" + str(scan_idx) + "_naomi" + ".png",
    #     dpi=300,
    #     bbox_inches="tight",
    # )
    # plt.close()

pd.to_pickle(mod_df_ct, 'mod_df_ct.pkl')

In [ ]:
#### Plot all the results combined 

df = mod_df_ct.copy()

df['depth_values'] = df.depth_values.values + np.random.normal(0, 15, df.depth_values.values.shape)

g = sns.relplot(
    y="depth_values", 
    x="correlation_values", 
    hue='Cell Type', 
    # orient='h', 
    # kind="strip", 
    # dodge=True, 
    alpha=0.5,
    s=40,
    # jitter=0.3,
    col='session_scan',
    col_wrap=5,
    kind='scatter',
    data=df)


g.set_axis_labels("Predictive Correlation", "Depth")

for ax in g.axes:
    ax.invert_yaxis()
    ax.set_title(f'Session {ax.get_title()[-3]}, Scan {ax.get_title()[-1]}')

savedir = "/home/naomi/Desktop/code/microns_rahul/saves/"
g.savefig(
    savedir + "boxplot_predcorr_celltype_ALL_naomi" + ".png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
